# 03 - Khai Phá Tri Thức: Luật Kết Hợp & Phân Cụm

**Mục tiêu:**
- Áp dụng Apriori tìm luật kết hợp liên quan bệnh tim
- Phân cụm KMeans & Hierarchical
- Xác định số cụm tối ưu
- Mô tả đặc điểm từng cụm nguy cơ

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

from src.data.loader import load_params
params = load_params()
seed = params['seed']

## 3.1 Luật Kết Hợp (Association Rule Mining - Apriori)

In [ ]:
# Load dữ liệu đã rời rạc hoá
df_disc = pd.read_csv(params['paths']['discretized_data'])
print(f"Shape: {df_disc.shape}")
print(f"Columns: {list(df_disc.columns)}")
df_disc.head()

In [ ]:
from src.mining.association import run_apriori, format_rules_table, interpret_rules

freq_items, rules, rules_heart = run_apriori(
    df_disc,
    min_support=params['apriori']['min_support'],
    min_confidence=params['apriori']['min_confidence'],
    min_lift=params['apriori']['min_lift'],
)

In [ ]:
# Top frequent itemsets
print("Top 15 Frequent Itemsets:")
freq_items.head(15)

In [ ]:
# Luật kết hợp liên quan bệnh tim
rules_table = format_rules_table(rules_heart, top_n=15)
rules_table

In [ ]:
# Diễn giải luật theo ý nghĩa y học
interpretations = interpret_rules(rules_heart, top_n=10)
for interp in interpretations:
    print(interp)
    print()

In [ ]:
# Lưu kết quả
from src.evaluation.report import save_rules_table
if len(rules_table) > 0:
    save_rules_table(rules_table, 'association_rules_heart')

In [ ]:
# Visualize: Support vs Confidence
if len(rules) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].scatter(rules['support'], rules['confidence'], 
                    c=rules['lift'], cmap='RdYlGn', alpha=0.7, s=50)
    axes[0].set_xlabel('Support')
    axes[0].set_ylabel('Confidence')
    axes[0].set_title('Support vs Confidence (color=Lift)')
    plt.colorbar(axes[0].collections[0], ax=axes[0], label='Lift')
    
    if len(rules_heart) > 0:
        axes[1].scatter(rules_heart['support'], rules_heart['confidence'],
                        c=rules_heart['lift'], cmap='RdYlGn', alpha=0.7, s=50)
        axes[1].set_xlabel('Support')
        axes[1].set_ylabel('Confidence')
        axes[1].set_title('Heart Disease Rules: Support vs Confidence')
        plt.colorbar(axes[1].collections[0], ax=axes[1], label='Lift')
    
    plt.tight_layout()
    plt.savefig('outputs/figures/association_rules.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3.2 Phân Cụm (Clustering)

In [ ]:
# Load processed data
df_clean = pd.read_csv(params['paths']['processed_data'])

from src.features.builder import select_features_for_modeling
X, y = select_features_for_modeling(df_clean)

# Chuẩn hoá
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"X_scaled shape: {X_scaled.shape}")

### 3.2.1 Tìm số cụm tối ưu (Elbow + Silhouette)

In [ ]:
from src.mining.clustering import find_optimal_k
from src.visualization.plots import plot_elbow_silhouette

k_range = range(params['clustering']['k_range_min'], params['clustering']['k_range_max'] + 1)
k_results = find_optimal_k(X_scaled, k_range, seed)

In [ ]:
fig = plot_elbow_silhouette(k_results['k_range'], k_results['inertias'], k_results['silhouettes'])
plt.show()

### 3.2.2 KMeans Clustering

In [ ]:
from src.mining.clustering import run_kmeans, profile_clusters, get_cluster_insights

best_k = k_results['best_k']
kmeans_labels, kmeans_model, kmeans_sil, kmeans_dbi = run_kmeans(X_scaled, best_k, seed)

In [ ]:
# Profile clusters
profile = profile_clusters(df_clean, kmeans_labels)

In [ ]:
# Cluster insights
insights = get_cluster_insights(df_clean, kmeans_labels)
for insight in insights:
    print(insight)

### 3.2.3 Hierarchical Clustering

In [ ]:
from src.mining.clustering import run_hierarchical
from scipy.cluster.hierarchy import dendrogram, linkage

Z = linkage(X_scaled[:100], method='ward')

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(Z, ax=ax, truncate_mode='level', p=5)
ax.set_title('Hierarchical Clustering Dendrogram', fontsize=14, fontweight='bold')
ax.set_xlabel('Samples')
ax.set_ylabel('Distance')
plt.tight_layout()
plt.savefig('outputs/figures/dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
hier_labels, hier_model, hier_sil, hier_dbi = run_hierarchical(X_scaled, best_k)

print(f"\n=== SO SÁNH CLUSTERING ===")
comparison_df = pd.DataFrame({
    'KMeans': [kmeans_sil, kmeans_dbi],
    'Hierarchical': [hier_sil, hier_dbi]
}, index=['Silhouette Score (cao=tốt)', 'Davies-Bouldin Index (thấp=tốt)'])
print(comparison_df.round(4))

### 3.2.4 Visualization Clusters

In [ ]:
from src.visualization.plots import plot_cluster_profiles

df_cluster = df_clean.copy()
df_cluster['cluster'] = kmeans_labels

fig = plot_cluster_profiles(df_cluster)
plt.show()

In [ ]:
# Scatter plot 2D (PCA)
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KMeans
scatter1 = axes[0].scatter(X_pca[:,0], X_pca[:,1], c=kmeans_labels, cmap='Set2', alpha=0.6, s=30)
axes[0].set_title(f'KMeans (k={best_k}, Sil={kmeans_sil:.3f})')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# By target
scatter2 = axes[1].scatter(X_pca[:,0], X_pca[:,1], c=y, cmap='RdYlGn', alpha=0.6, s=30)
axes[1].set_title('Colored by Target (Disease)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
plt.colorbar(scatter2, ax=axes[1], label='Target')

plt.suptitle('PCA Visualization', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.3 Nhận xét

**Luật kết hợp:**
- Apriori tìm được các tổ hợp triệu chứng liên quan bệnh tim
- Các yếu tố xuất hiện nhiều: đau ngực không triệu chứng, ST depression, cholesterol cao

**Phân cụm:**
- Số cụm tối ưu xác định bằng Silhouette Score
- Mỗi cụm có đặc trưng và tỷ lệ bệnh khác nhau
- Nhóm nguy cơ cao được xác định rõ ràng